# Analisis Feature Importance dalam Prediksi Hasil Pertandingan Sepak Bola
# Menggunakan Explainable Machine Learning
**Dataset:** European Soccer Database (Kaggle) | **Model:** Random Forest | **XAI:** SHAP

## 1. Mount Drive & Install Library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install shap --quiet

## 2. Import Library

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score,
    precision_score, recall_score
)
print('✅ Library berhasil diimport!')

## 3. Load Dataset

In [ ]:
# =====================================================
# Sesuaikan path dengan lokasi file di Drive kamu
# =====================================================
db_path = '/content/drive/MyDrive/dataset/database.sqlite'

# Buka koneksi BARU (jangan close dulu sampai query selesai)
conn = sqlite3.connect(db_path)

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)
print('Tabel yang tersedia:')
print(tables)

## 4. Query Data (Subquery — Anti Duplikasi)

In [ ]:
# =====================================================
# Subquery memastikan hanya 1 baris atribut tim
# per pertandingan (atribut terbaru sebelum match)
# → mencegah data duplikat 156.000 baris
# =====================================================
query = """
SELECT
    m.id            AS match_id,
    m.season,
    m.date          AS match_date,
    m.home_team_goal,
    m.away_team_goal,

    hta.buildUpPlaySpeed        AS home_buildUpPlaySpeed,
    hta.buildUpPlayDribbling    AS home_buildUpPlayDribbling,
    hta.buildUpPlayPassing      AS home_buildUpPlayPassing,
    hta.chanceCreationPassing   AS home_chanceCreationPassing,
    hta.chanceCreationCrossing  AS home_chanceCreationCrossing,
    hta.chanceCreationShooting  AS home_chanceCreationShooting,
    hta.defencePressure         AS home_defencePressure,
    hta.defenceAggression       AS home_defenceAggression,
    hta.defenceTeamWidth        AS home_defenceTeamWidth,

    ata.buildUpPlaySpeed        AS away_buildUpPlaySpeed,
    ata.buildUpPlayDribbling    AS away_buildUpPlayDribbling,
    ata.buildUpPlayPassing      AS away_buildUpPlayPassing,
    ata.chanceCreationPassing   AS away_chanceCreationPassing,
    ata.chanceCreationCrossing  AS away_chanceCreationCrossing,
    ata.chanceCreationShooting  AS away_chanceCreationShooting,
    ata.defencePressure         AS away_defencePressure,
    ata.defenceAggression       AS away_defenceAggression,
    ata.defenceTeamWidth        AS away_defenceTeamWidth

FROM Match m

JOIN Team_Attributes hta
    ON hta.id = (
        SELECT id FROM Team_Attributes
        WHERE team_api_id = m.home_team_api_id
          AND date <= m.date
        ORDER BY date DESC
        LIMIT 1
    )

JOIN Team_Attributes ata
    ON ata.id = (
        SELECT id FROM Team_Attributes
        WHERE team_api_id = m.away_team_api_id
          AND date <= m.date
        ORDER BY date DESC
        LIMIT 1
    )

WHERE
    m.home_team_goal IS NOT NULL
    AND m.away_team_goal IS NOT NULL
"""

print('Menjalankan query... (estimasi 30-90 detik)')
df = pd.read_sql(query, conn)

# Tutup koneksi setelah query selesai
conn.close()

print(f'✅ Jumlah data: {len(df)} pertandingan')
print(f'   (Seharusnya ~21.000 – 25.000)')
print(f'\nKolom: {df.columns.tolist()}')
print()
print(df.head())

## 5. Preprocessing

In [ ]:
# Label target
label_map = {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}

def get_result(row):
    if row['home_team_goal'] > row['away_team_goal']:
        return 2   # Home Win
    elif row['home_team_goal'] < row['away_team_goal']:
        return 0   # Away Win
    else:
        return 1   # Draw

df['result'] = df.apply(get_result, axis=1)

print('Distribusi label:')
dist = df['result'].value_counts().rename(index=label_map)
print(dist)

plt.figure(figsize=(6, 4))
dist.plot(kind='bar', color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='black')
plt.title('Distribusi Hasil Pertandingan', fontsize=14)
plt.xlabel('Hasil')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
features = [
    'home_buildUpPlaySpeed', 'home_buildUpPlayDribbling', 'home_buildUpPlayPassing',
    'home_chanceCreationPassing', 'home_chanceCreationCrossing', 'home_chanceCreationShooting',
    'home_defencePressure', 'home_defenceAggression', 'home_defenceTeamWidth',
    'away_buildUpPlaySpeed', 'away_buildUpPlayDribbling', 'away_buildUpPlayPassing',
    'away_chanceCreationPassing', 'away_chanceCreationCrossing', 'away_chanceCreationShooting',
    'away_defencePressure', 'away_defenceAggression', 'away_defenceTeamWidth',
]

X = df[features].copy()
y = df['result'].copy()

print(f'Jumlah fitur  : {len(features)}')
print(f'Jumlah sampel : {len(X)}')

In [ ]:
# Handling missing values
print('Missing values sebelum:')
print(X.isnull().sum())

X = X.fillna(X.median())

print(f'\n✅ Missing values setelah: {X.isnull().sum().sum()}')

## 6. Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {len(X_train)} sampel')
print(f'Test  : {len(X_test)} sampel')
print(f'\nDistribusi kelas Train:')
print(y_train.value_counts().rename(index=label_map))

## 7. Training Model

In [ ]:
# class_weight='balanced' mengatasi class imbalance
# agar Draw dan Away Win ikut dipelajari dengan baik
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

model.fit(X_train, y_train)
print('✅ Model berhasil dilatih!')

## 8. Evaluasi Model

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print('=' * 45)
print('           HASIL EVALUASI MODEL')
print('=' * 45)
print(f'Accuracy : {acc:.4f} ({acc*100:.2f}%)')
print()
print('Classification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=['Away Win', 'Draw', 'Home Win']
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
labels = ['Away Win', 'Draw', 'Home Win']

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Random Forest', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Ranking Feature Importance:')
print(feat_imp.to_string(index=False))

colors = ['#2ecc71' if 'home' in f else '#e74c3c' for f in feat_imp['Feature']]
plt.figure(figsize=(10, 7))
plt.barh(feat_imp['Feature'][::-1], feat_imp['Importance'][::-1],
         color=colors[::-1], edgecolor='black')
plt.title('Feature Importance - Random Forest\n(Hijau=Home | Merah=Away)', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 10. SHAP Analysis

In [ ]:
print('Menghitung SHAP values...')
X_shap = X_test.iloc[:500]

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_shap)

print('✅ SHAP values selesai!')
print(f'Shape: {shap_values.values.shape}  (sampel x fitur x kelas)')

In [ ]:
print('=== SHAP Bar Plot ===')
shap.plots.bar(shap_values, max_display=18)

In [ ]:
print('=== SHAP Beeswarm Plot (kelas: Home Win) ===')
shap.plots.beeswarm(shap_values[:, :, 2], max_display=18)

In [ ]:
print('=== SHAP Waterfall Plot (sampel ke-0) ===')
print(f'Prediksi : {label_map[int(model.predict(X_shap.iloc[[0]])[0])]}')
print(f'Asli     : {label_map[int(y_test.iloc[0])]}')
shap.plots.waterfall(shap_values[0, :, 2])

## 11. Ringkasan Akhir

In [ ]:
f1  = f1_score(y_test, y_pred, average='weighted')
pre = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')

print('=' * 50)
print('      RINGKASAN HASIL UNTUK JURNAL')
print('=' * 50)
print(f'Dataset      : European Soccer Database (Kaggle)')
print(f'Jumlah Data  : {len(df)} pertandingan')
print(f'Jumlah Fitur : {len(features)} fitur atribut tim')
print(f'Model        : Random Forest (n=200, depth=10)')
print(f'Class Weight : balanced')
print(f'Split        : 80% Train / 20% Test')
print('-' * 50)
print(f'Accuracy     : {acc:.4f} ({acc*100:.2f}%)')
print(f'Precision    : {pre:.4f}')
print(f'Recall       : {rec:.4f}')
print(f'F1-Score     : {f1:.4f}')
print('-' * 50)
print('Top 5 Feature Importance:')
for i, row in feat_imp.head(5).iterrows():
    rank = feat_imp.index.tolist().index(i) + 1
    print(f'  {rank}. {row["Feature"]} ({row["Importance"]:.4f})')
print('=' * 50)